# LangChain Prompt Templates and Automation

In this notebook, I practiced using LangChain PromptTemplate to create reusable prompts.

The goal is to automate two LLM tasks:

1. Summarizing a Markdown document.
2. Extracting keywords from a product description.

This notebook is part of Week 5 Day 3: Prompt Templates and Automation using LangChain.

In [1]:
from pathlib import Path

from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama


## 1. Define File Paths

The input text is stored outside the code in separate files.

This makes the workflow more realistic because real LLM applications usually read documents, product descriptions, or user inputs from external sources.

In [2]:
BASE_DIR = Path(".")

MARKDOWN_FILE = BASE_DIR / "sample_markdown_doc.md"
PRODUCT_FILE = BASE_DIR / "product_description.txt"

OUTPUT_DIR = BASE_DIR / "reports" / "llm_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MARKDOWN_FILE, PRODUCT_FILE, OUTPUT_DIR

(WindowsPath('sample_markdown_doc.md'),
 WindowsPath('product_description.txt'),
 WindowsPath('reports/llm_outputs'))

## 2. Connect to the Local LLM

I used Ollama with a local model.

The model receives the final prompt after LangChain fills the template variables.

In [3]:
llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0.2
)

## 3. Create Prompt Template for Markdown Summarization

This template has two variables:

- document_title
- markdown_text

LangChain will replace these variables with real values before sending the prompt to the LLM.

In [4]:
summary_template = PromptTemplate(
    input_variables=["document_title", "markdown_text"],
    template="""
You are a helpful AI assistant.

Your task is to summarize the following markdown document.

Document title:
{document_title}

Markdown content:
{markdown_text}

Return the answer in this format:

Summary:
- Point 1
- Point 2
- Point 3

Main idea:
Write one short sentence explaining the main idea of the document.
"""
)

## 4. Create Prompt Template for Keyword Extraction

This template has two variables:

- product_name
- product_description

The goal is to extract useful keywords from a product description.

In [5]:
keyword_template = PromptTemplate(
    input_variables=["product_name", "product_description"],
    template="""
You are a product analysis assistant.

Your task is to extract useful keywords from the product description.

Product name:
{product_name}

Product description:
{product_description}

Return the answer in this format:

Keywords:
- keyword 1
- keyword 2
- keyword 3
- keyword 4
- keyword 5

Product category:
Write the most likely product category.

Short explanation:
Briefly explain why these keywords are useful.
"""
)

## 5. Read Input Files

Instead of writing the input text directly inside the code, the notebook reads text from external files.

In [6]:
def read_file(file_path):
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    
    return file_path.read_text(encoding="utf-8")

In [7]:
markdown_text = read_file(MARKDOWN_FILE)

markdown_text

'# Prompt Templates in LangChain\n\nPrompt templates are reusable prompt structures that help developers communicate with large language models in a clear and organized way.\n\nInstead of writing the same prompt again and again, developers can create a template with variables.\n\nFor example, a template can include variables such as user question, context, document text, or product description.\n\nLangChain makes this easier by providing PromptTemplate objects.\n\nPrompt templates are useful because they make LLM applications more consistent, reusable, and easier to automate.\n'

In [8]:
product_description = read_file(PRODUCT_FILE)

product_description

'These wireless headphones provide high-quality sound, active noise cancellation,\ncomfortable ear cushions, Bluetooth connectivity, and up to 30 hours of battery life.\nThey are designed for travel, work, studying, and daily music listening.'

## 6. Run Markdown Summarization Pipeline

The pipeline is:

Markdown file → PromptTemplate → LLM → Summary output

In [9]:
summary_input = {
    "document_title": "Prompt Templates in LangChain",
    "markdown_text": markdown_text
}

summary_prompt = summary_template.format(**summary_input)

summary_response = llm.invoke(summary_prompt)

summary_output = summary_response.content

print(summary_output)

Here is a summary of the markdown document:

Summary:
Prompt templates are reusable structures for communicating with large language models, making development more efficient and consistent.

Main idea:
The document explains how prompt templates can be used in LangChain to create clear and organized prompts for large language models.


In [10]:
summary_output_path = OUTPUT_DIR / "notebook_markdown_summary_output.txt"

summary_output_path.write_text(summary_output, encoding="utf-8")

summary_output_path

WindowsPath('reports/llm_outputs/notebook_markdown_summary_output.txt')

## 7. Run Keyword Extraction Pipeline

The pipeline is:

Product description file → PromptTemplate → LLM → Keyword output

In [11]:
keyword_input = {
    "product_name": "Wireless Noise-Cancelling Headphones",
    "product_description": product_description
}

keyword_prompt = keyword_template.format(**keyword_input)

keyword_response = llm.invoke(keyword_prompt)

keyword_output = keyword_response.content

print(keyword_output)

Keywords:
- wireless
- noise-cancelling
- headphones
- active
- Bluetooth
- comfort
- battery-life
- travel
- work
- studying
- daily
- music

Product category: Wireless Headphones

These keywords are useful because they describe the key features and benefits of the product, such as its wireless connectivity, noise-cancelling technology, comfortable design, and long battery life. They also highlight the product's intended use cases (travel, work, studying, etc.) and provide a clear understanding of what users can expect from the product.


In [12]:
keyword_output_path = OUTPUT_DIR / "notebook_keyword_extraction_output.txt"

keyword_output_path.write_text(keyword_output, encoding="utf-8")

keyword_output_path

WindowsPath('reports/llm_outputs/notebook_keyword_extraction_output.txt')

## 8. What I Learned

In this task, I learned that prompt templates are reusable prompt structures.

Instead of writing a full prompt manually every time, I can define a template with variables and let Python fill the variables automatically.

This is useful because it makes LLM workflows more organized, reusable, and easier to automate.

The main difference from the previous tasks is that Task 1 and Task 2 focused on writing prompts manually, while this task focused on using LangChain to automate prompts in code.